# Outcome definitions and commensurability

This offline orientation notebook compares four declared outcomes with Isoprax's structured comparator. It does not load benchmark rows, measure predictor quality, or pool family-level results. C-MAPSS is represented here by its RUL outcome; its run-to-failure definition remains separately available in the C-MAPSS module.

The displayed matrix is a human-review signal derived from `check_commensurable()`. Only a comparator result with `pooling_allowed=True` permits pooling under the current declarations. Calibration does not repair an event or observation-process mismatch.

In [ ]:
import os
import sys
from pathlib import Path

working_directory = Path.cwd().resolve()
working_root = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "pyproject.toml").is_file() and (candidate / "isoprax").is_dir()
    ),
    None,
)
configured_root = os.environ.get("ISOPRAX_REPO_ROOT")
REPO_ROOT = (
    Path(configured_root).expanduser().resolve() if configured_root else working_root
)
if (
    REPO_ROOT is None
    or not (REPO_ROOT / "pyproject.toml").is_file()
    or not (REPO_ROOT / "isoprax").is_dir()
):
    raise RuntimeError(
        "Launch from this checkout or set ISOPRAX_REPO_ROOT to its repository root."
    )
if configured_root and working_root is not None and REPO_ROOT != working_root:
    raise RuntimeError(
        "ISOPRAX_REPO_ROOT does not match the notebook's checkout directory."
    )
repo_path = str(REPO_ROOT)
if repo_path in sys.path:
    sys.path.remove(repo_path)
sys.path.insert(0, repo_path)

import isoprax

PACKAGE_ROOT = Path(isoprax.__file__).resolve().parents[1]
if PACKAGE_ROOT != REPO_ROOT:
    raise RuntimeError(
        "The selected Python kernel does not import Isoprax from this checkout."
    )

from IPython.display import display

from isoprax.ai4i2020 import ai4i2020_outcome_definitions
from isoprax.apachejit import apachejit_outcome_definition
from isoprax.commensurability import check_commensurable
from isoprax.metropt3 import metropt3_outcome_definition
from isoprax.nasa_cmaps import cmapss_outcome_definitions

print(f"Python: {sys.version.split()[0]} ({sys.executable})")
print(f"Repository: {REPO_ROOT}")

In [ ]:
definitions = {
    "AI4I 2020 — composite machine failure": ai4i2020_outcome_definitions()[
        "Machine failure"
    ],
    "ApacheJIT — buggy repository commit": apachejit_outcome_definition(),
    "NASA C-MAPSS — remaining useful life": cmapss_outcome_definitions()["rul"],
    "MetroPT-3 — externally reported air leak": metropt3_outcome_definition(),
}
definition_rows = [
    {"dataset outcome": name, **definition.to_dict()}
    for name, definition in definitions.items()
]
display(definition_rows)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

labels = list(definitions)
level_code = {"irreducible": 0, "bridgeable": 1, "direct": 2, "attested": 3}
level_colors = ["#b84a4a", "#e0a33a", "#4d9568", "#3977a8"]
matrix = np.zeros((len(labels), len(labels)), dtype=int)
comparison_rows = []

for row_index, (left_name, left) in enumerate(definitions.items()):
    for column_index, (right_name, right) in enumerate(definitions.items()):
        result = check_commensurable(left, right)
        matrix[row_index, column_index] = level_code[result.level]
        comparison_rows.append(
            {
                "left": left_name,
                "right": right_name,
                "level": result.level,
                "differing_fields": result.differing_fields,
                "pooling_allowed": result.pooling_allowed,
                "reason": result.reason,
            }
        )

display(comparison_rows)
figure, axis = plt.subplots(figsize=(11, 8), constrained_layout=True)
cmap = ListedColormap(level_colors)
norm = BoundaryNorm(np.arange(cmap.N + 1) - 0.5, cmap.N)
axis.imshow(matrix, cmap=cmap, norm=norm)
axis.set_xticks(range(len(labels)), labels=labels, rotation=35, ha="right")
axis.set_yticks(range(len(labels)), labels=labels)
axis.set_title("Structured outcome commensurability (core comparator)")
for row_index, (left_name, left) in enumerate(definitions.items()):
    for column_index, (right_name, right) in enumerate(definitions.items()):
        result = check_commensurable(left, right)
        annotation = f"{result.level}\n" + (
            "pool allowed" if result.pooling_allowed else "no pooling"
        )
        axis.text(
            column_index, row_index, annotation, ha="center", va="center", fontsize=8
        )
axis.set_xlabel("Compared outcome definition")
axis.set_ylabel("Outcome definition")
axis.legend(
    handles=[
        Patch(color=color, label=label)
        for color, label in zip(
            level_colors,
            ("irreducible", "bridgeable", "direct", "attested"),
            strict=True,
        )
    ],
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
)
display(figure)
plt.close(figure)

## Interpretation boundary

A green diagonal is only self-comparison. Off-diagonal levels and pooling decisions come from the structured definitions and core comparator; they are not empirical validation, model performance, or a license to combine raw labels. Review the `differing_fields` and `reason` columns before making any comparison claim.